# AStrategySARSAJ

> JAX CRLD SARSA agents in strategy space. Inherits from strategybaseJ.

In [ ]:
#| default_exp Agents/StrategySARSAJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import itertools as it
from functools import partial

import jax
from jax import jit
import jax.numpy as jnp

from fastcore.utils import *

from pyCRLD.Agents.StrategyBaseJ import strategybaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class stratSARSAJ(strategybaseJ):
    """Class for JAX CRLD SARSA agents in strategy space."""

    @partial(jit, static_argnums=(0, 2))
    def RPEisa(self,
               Xisa,       # Joint strategy
               norm=False  # normalize error around actions?
               ) -> jnp.ndarray:  # RP/TD error
        """Reward-prediction error for SARSA dynamics, given joint strategy `Xisa`."""
        R = self.Risa(Xisa)
        NextQ = self.NextQisa(Xisa, Risa=R)

        n = jnp.newaxis
        E = (self.pre[:, n, n] * R
             + self.gamma[:, n, n] * NextQ
             - 1 / self.beta[:, n, n] * jnp.log(Xisa))
        E *= self.beta[:, n, n]

        E = E - E.mean(axis=2, keepdims=True) if norm else E
        return E

    @partial(jit, static_argnums=0)
    def NextQisa(self,
                 Xisa,
                 Qisa=None,
                 Risa=None,
                 Vis=None,
                 Tisas=None) -> jnp.ndarray:
        """Strategy-average next state-action value for agent i, state s, action a."""
        Qisa = self.Qisa(Xisa) if Qisa is None else Qisa

        i = 0; a = 1; s = 2; s_ = 3
        j2k = list(range(6, 6 + self.N - 1))
        b2d = list(range(6 + self.N - 1, 6 + self.N - 1 + self.N))
        e2f = list(range(5 + 2 * self.N, 5 + 2 * self.N + self.N - 1))

        sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
        otherX = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))

        NextQis = jnp.einsum(Qisa, [i, s_, a], Xisa, [i, s_, a], [i, s_])

        args = [self.Omega, [i] + j2k + [a] + b2d + e2f] + otherX + \
               [self.T, [s] + b2d + [s_], NextQis, [i, s_], [i, s, a]]
        return jnp.einsum(*args, optimize=self.opti)

## Tests

In [ ]:
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
agent = stratSARSAJ(env, learning_rates=0.1, discount_factors=0.9, choice_intensities=2.0)

X = agent.zero_intelligence_strategy()
rpe = agent.RPEisa(X)
assert rpe.shape == (2, 1, 2)

In [ ]:
key = jax.random.PRNGKey(1)
X0 = agent.random_softmax_strategy(key=key)
traj, fpr = agent.trajectory(X0, Tmax=30)
assert traj.shape == (30, 2, 1, 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()